# Análisis Interactivo: Replicación SSCAD 2025 en Clúster Lenovo P3 Tiny

Este notebook permite cargar interactivamente los datos experimentales obtenidos de la ejecución de **Task Bench** con **LLVM MPI Proxy Plugin (MPP)** y **MPI + OpenMP**, calculando métricas de:
- **Rendimiento Pico y TFLOP/s**
- **METG(50%) (Minimum Effective Task Granularity)**
- **Weak Scaling (1 a 10 nodos)**
- **Comparativa directa frente a NVIDIA H100 y AMD MI300A**


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.append("..")
sys.path.append("../scripts")
import utils

# Configuración de estilo de gráficos
sns.set_theme(style="whitegrid", font_scale=1.1)
DATA_DIR = Path("../results")
FIGURES_DIR = Path("../figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## 1. Carga y Procesamiento de Datos Experimentales

In [ ]:
import json

jsonl_file = DATA_DIR / "tb_wscaling_stencil_5s.jsonl"
if not jsonl_file.exists():
    jsonl_file = Path("../../out/tb_wscaling_stencil_5s.jsonl")

records = []
with open(jsonl_file, "r") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Total registros cargados: {len(df)}")
df.head()

## 2. Cálculo de Granularidad de Tarea y Eficiencia

In [ ]:
agg_df = df.groupby(by=["sys_type", "workers", "size", "total_flops", "total_tasks"]).agg(
    runtime_mean=("wall_time", "mean"),
    runtime_std=("wall_time", "std")
).reset_index()

agg_df["tflops_per_sec"] = agg_df["total_flops"] * 1e-12 / agg_df["runtime_mean"]
agg_df["task_granularity"] = agg_df["runtime_mean"] * agg_df["workers"] * 1e3 / agg_df["total_tasks"]

# Eficiencia relativa al pico sostenido por configuración
max_flops = agg_df.groupby("workers")["tflops_per_sec"].transform("max")
agg_df["efficiency"] = agg_df["tflops_per_sec"] * 100 / max_flops
agg_df.head(10)

## 3. Gráfica de Granularidad vs Eficiencia (METG 50%)

In [ ]:
plt.figure(figsize=(10, 6))
for sys_name, group in agg_df[agg_df["workers"] == 10].groupby("sys_type"):
    group_sorted = group.sort_values("task_granularity")
    plt.plot(group_sorted["task_granularity"], group_sorted["efficiency"], marker="o", label=sys_name.upper())

plt.axhline(50, color="red", linestyle="--", label="Umbral METG (50%)")
plt.xscale("log")
plt.xlabel("Task Granularity (ms)")
plt.ylabel("Eficiencia (% del Pico)")
plt.title("Eficiencia vs Granularidad de Tarea (10 Nodos RTX A1000)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "metg_a1000_10nodos.png", dpi=300)
plt.show()